# Chapter 13 — Text as Data
### Companion Notebook

*Data Science for Business and Economics: A Python Approach* — Dr. Jorge Zazueta Gutiérrez

This notebook runs every code example in Chapter 13, in the order of the chapter, and draws its four figures. Ana Delgado, a customer-experience analyst at Northstar Mobile, has 9,105 support messages written by 5,079 customers between July 2023 and December 2024, and a churn table that says which customers left. The messages are synthetic: they were written by the program `scripts/ch13_generate_messages.py` in the companion repository, so the events behind them are known and every finding can be checked against them.

All data are read from the companion repository, so the notebook needs an internet connection. Every cell is deterministic: run from top to bottom, it prints the numbers in the chapter. The topic model and the cross-validated churn model take a minute or two each.

In [ ]:
book_metadata = {
    "title": "Data Science for Business and Economics: A Python Approach",
    "chapter": 13,
    "chapter_title": "Text as Data",
    "author": "Jorge Zazueta Gutierrez",
    "protagonist": "Ana Delgado",
    "part": 2,
    "part_title": "Methods and Models",
    "requires": ["pandas", "numpy", "pyarrow", "scikit-learn", "vaderSentiment",
                 "matplotlib", "seaborn"],
    "notebook_version": "0.1",
}
book_metadata

## Setup

VADER is not preinstalled in Colab. This cell installs it once per session; the other libraries are already there.

In [ ]:
# Install the VADER sentiment dictionary (once per Colab session)
%pip install -q vaderSentiment

In [ ]:
# Libraries used by the figure cells
import matplotlib.pyplot as plt
import seaborn as sns

# The book's figure style: light grid, steel blue for the highlight, gray for the rest
sns.set_theme(style="whitegrid",
              rc={"grid.color": "#EEEEEE", "axes.edgecolor": "#BBBBBB"})
BLUE, GRAY = "#4682B4", "#8C8C8C"

## 13.1 Nine Thousand Messages

### 13.1.1 Ana's Corpus

**Code Example 13.1.** Loading the customers and their messages.

In [ ]:
import pandas as pd

# Every file for the chapter lives in the companion repository
BASE = ("https://raw.githubusercontent.com/jzazueta/"
        "ds-business-economics/main/data/")

# The churn table: one row per customer, with the outcome
customers = pd.read_csv(BASE + "ch13_customers.csv")

# The support inbox: one row per message
msgs = pd.read_parquet(BASE + "ch13_messages.parquet")
print(customers.shape, msgs.shape)
print(msgs["customer_id"].nunique(), "customers wrote at least once")
print(msgs["channel"].value_counts())
print(msgs["date"].min().date(), "to", msgs["date"].max().date())

**Code Example 13.2.** How long the messages are, and three of them.

In [ ]:
# Count words by splitting each message on white space
words = msgs["text"].str.split().str.len()
print("total words:", words.sum())
print(words.describe().round(1))

# Three messages drawn at random
for t in msgs["text"].sample(3, random_state=4):
    print("-", t)

**Code Example 13.3.** Attaching the churn outcome to each message.

In [ ]:
msgs = msgs.merge(customers[["customer_id", "churned"]],
                  on="customer_id")
print(customers["churned"].mean().round(3))

# Churn among customers who wrote and customers who did not
wrote = customers["customer_id"].isin(msgs["customer_id"])
print(customers.groupby(wrote)["churned"].mean().round(3))

### 13.1.2 From Text to Measurement

## 13.2 From Text to Numbers

### 13.2.1 Tokens, Vocabulary, and the Document-Term Matrix

**Code Example 13.4.** A document-term matrix for three messages.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = ["My internet is down again.",
        "I was charged twice, please refund the charge.",
        "Thanks, the internet works again!"]

# Learn the vocabulary, then count each word in each message
vec = CountVectorizer()
X = vec.fit_transform(docs)
print(vec.get_feature_names_out())
print(X.toarray())

### 13.2.2 The Whole Inbox

**Code Example 13.5.** The document-term matrix of the inbox.

In [ ]:
import numpy as np

vec = CountVectorizer()
X = vec.fit_transform(msgs["text"])
# Rows, columns, and the share of cells that are not zero
print(X.shape, f"{X.nnz / (X.shape[0] * X.shape[1]):.4f}")

# The twelve most frequent words in the whole inbox
totals = pd.Series(np.asarray(X.sum(axis=0)).ravel(),
                   index=vec.get_feature_names_out())
print(totals.sort_values(ascending=False).head(12).to_string())

**Code Example 13.6.** The most frequent words once stop words are removed.

In [ ]:
vec_sw = CountVectorizer(stop_words="english")
X_sw = vec_sw.fit_transform(msgs["text"])
totals_sw = pd.Series(np.asarray(X_sw.sum(axis=0)).ravel(),
                      index=vec_sw.get_feature_names_out())
print(X_sw.shape)
top = totals_sw.sort_values(ascending=False).head(15)
print(top.to_string())

### 13.2.3 Counting Over Time

**Code Example 13.7.** Share of messages that mention a word, by month.

In [ ]:
text = msgs["text"].str.lower()
month = msgs["date"].dt.to_period("M")

# True or False for each message; the pattern catches spellings
# such as "velofiber", "velo fiber" and "velofibre"
flags = pd.DataFrame({
    "outage": text.str.contains("outage"),
    "velofiber": text.str.contains(r"velo ?fib(?:er|re)"),
    "cancel": text.str.contains("cancel"),
})
share = flags.groupby(month).mean().round(3)
print(share.T.iloc[:, [2, 3, 4, 10, 11, 12]].to_string())
print(text.str.extract(r"(velo ?fib(?:er|re))",
                       expand=False).value_counts())

**Figure 13.1 — share of each month's messages that mention three words.** Not a code example in the printed chapter: this cell draws the figure from the variables above.

In [ ]:
# Percent of each month's messages that mention each word
pct = flags.groupby(month).mean() * 100
x = pct.index.to_timestamp()                     # months as dates, for the axis

fig, ax = plt.subplots(figsize=(10, 4.3))
for col, label, color in [("outage", '"outage"', GRAY),
                          ("velofiber", '"VeloFiber" (any spelling)', BLUE),
                          ("cancel", '"cancel"', "#333333")]:
    ax.plot(x, pct[col], marker="o", ms=3.5, lw=2, color=color)
    # Label each line at its right end instead of using a legend
    ax.text(x[-1] + pd.Timedelta(days=12), pct[col].iloc[-1], label,
            va="center", fontsize=9, color=color)

# Mark the two known events
ax.annotate("network outage,\nOctober 14, 2023",
            xy=(pd.Timestamp("2023-10-01"), pct["outage"].iloc[3]),
            xytext=(pd.Timestamp("2023-12-10"), 26), fontsize=8.5, color="#555555",
            arrowprops=dict(arrowstyle="-", color="#999999", lw=0.8))
ax.annotate("VeloFiber enters\nthe market, June 2024",
            xy=(pd.Timestamp("2024-06-01"), pct["velofiber"].iloc[11]),
            xytext=(pd.Timestamp("2024-02-15"), 20), fontsize=8.5, color=BLUE,
            arrowprops=dict(arrowstyle="-", color=BLUE, lw=0.8))

ax.set_ylabel("% of messages that mention the word")
ax.set_xlim(x[0] - pd.Timedelta(days=15), x[-1] + pd.Timedelta(days=120))
ax.set_ylim(0, 32)
ticks = pd.date_range("2023-07-01", "2024-12-01", freq="3MS")
ax.set_xticks(ticks)
ax.set_xticklabels([t.strftime("%b %Y") for t in ticks])
ax.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
plt.show()

### 13.2.4 Reading the Words in Context

**Code Example 13.8.** The October messages that mention the outage.

In [ ]:
oct_outage = msgs.loc[text.str.contains("outage")
                      & (month == pd.Period("2023-10")), "text"]
print(len(oct_outage))
# The first 60 characters of each message, and how often each repeats
print(oct_outage.str[:60].value_counts().head(5).to_string())

## 13.3 Dictionaries and Sentiment

### 13.3.1 A General-Purpose Dictionary

**Code Example 13.9.** VADER scores for three sentences.

In [ ]:
# !pip install -q vaderSentiment       (once per Colab session)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
for t in ["Thanks for fixing the problem so quickly.",
          "My internet is down again.",
          "NorthPay charged me twice, please refund "
          "the duplicate payment. Thanks!"]:
    print(sia.polarity_scores(t)["compound"], "|", t)

**Code Example 13.10.** VADER scores for all messages, by churn.

In [ ]:
msgs["vader"] = [sia.polarity_scores(t)["compound"]
                 for t in msgs["text"]]
print(msgs.groupby("churned")["vader"].agg(["mean", "median"]).round(3))

# Double-charge messages: how many does VADER call positive?
twice = msgs["text"].str.contains("twice|duplicate", case=False)
positive = (msgs.loc[twice, "vader"] > 0.05).mean().round(3)
print("double-charge messages:", twice.sum(),
      " scored positive:", positive)

### 13.3.2 A Dictionary for Northstar

**Code Example 13.11.** A complaint list written for Northstar.

In [ ]:
complaint = ["twice", "duplicate", "double", "refund",
             "overdrawn", "outage", "no service", "no signal",
             "dropping", "down", "slow", "buffering",
             "throttled", "expensive", "fees", "afford",
             "cancel", "switch", "velofiber", "velo fiber",
             "leaving"]
pattern = r"\b(?:" + "|".join(complaint) + r")\b"
msgs["complaints"] = msgs["text"].str.lower().str.count(pattern)

print(msgs.groupby("churned")["complaints"].mean().round(3))
# Share of messages with at least one complaint term
print((msgs["complaints"] > 0).groupby(msgs["churned"]).mean().round(3))

**Figure 13.2 — VADER scores and the Northstar complaint list, by churn.** Not a code example in the printed chapter: this cell draws the figure from the variables above.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.2))

# Panel (a): distribution of VADER scores for each group of customers
for g, label, color in [(0, "customer stayed", GRAY), (1, "customer churned", BLUE)]:
    a.hist(msgs.loc[msgs["churned"] == g, "vader"], bins=np.linspace(-1, 1, 26),
           density=True, histtype="step", lw=2, color=color, label=label)
a.set_xlabel("VADER compound score (−1 negative, +1 positive)")
a.set_ylabel("density")
a.set_title("(a) General-purpose sentiment", loc="left", fontsize=11)
a.set_xticks([-1, -0.5, 0, 0.5, 1])
a.legend(frameon=False, fontsize=9, loc="upper left")

# Panel (b): number of complaint terms per message, 3 or more grouped together
n_terms = msgs["complaints"].clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: "3+"})
tab = pd.crosstab(n_terms, msgs["churned"], normalize="columns") * 100
xs = np.arange(len(tab))
b.bar(xs - 0.19, tab[0], 0.38, color=GRAY, label="customer stayed")
b.bar(xs + 0.19, tab[1], 0.38, color=BLUE, label="customer churned")
b.set_xticks(xs)
b.set_xticklabels(tab.index)
b.set_xlabel("complaint terms in the message")
b.set_ylabel("% of messages")
b.set_title("(b) Northstar complaint list", loc="left", fontsize=11)

for ax in (a, b):
    ax.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
plt.show()

## 13.4 Similarity: Finding Messages That Say the Same Thing

### 13.4.1 Weighting Words by How Rare They Are

**Code Example 13.12.** TF-IDF weights for the three messages.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
W = tfidf.fit_transform(docs)

# One row per word: its idf, then its weight in each message
table = pd.DataFrame(W.toarray().T, index=tfidf.get_feature_names_out(),
                     columns=["msg 0", "msg 1", "msg 2"])
table.insert(0, "idf", tfidf.idf_)
print(table.round(2).to_string())

### 13.4.2 Measuring How Close Two Messages Are

**Code Example 13.13.** Cosine similarity between the three messages.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Similarity of every message with every other: a 3 x 3 table
print(cosine_similarity(W).round(2))

### 13.4.3 Copies in the Inbox

**Code Example 13.14.** Near twins among the longer messages.

In [ ]:
# Ignore stop words and words used in fewer than 5 messages
tfidf = TfidfVectorizer(stop_words="english", min_df=5)
W = tfidf.fit_transform(msgs["text"])
print(W.shape)

# Messages of 20 words or more, compared with each other
long = msgs.index[msgs["text"].str.split().str.len() >= 20]
S = cosine_similarity(W[long])
np.fill_diagonal(S, 0)                    # ignore self-similarity
twins = pd.Series((S > 0.9).sum(axis=1), index=long)
print(len(long))
print(twins.value_counts().sort_index().to_string())
msgs["twins"] = twins.reindex(msgs.index).fillna(0).astype(int)

**Code Example 13.15.** The copies, and the outage count without them.

In [ ]:
copies = msgs[msgs["twins"] >= 20]
print(len(copies), copies["date"].min().date(),
      copies["date"].max().date())

# Share of messages that mention the outage, with and without copies
keep = msgs["twins"] < 20
o = text.str.contains("outage")
compare = pd.DataFrame({
    "all": o.groupby(month).mean(),
    "without copies": o[keep].groupby(month[keep]).mean()})
print(compare.round(3).T.iloc[:, 2:6].to_string())

### 13.4.4 Messages Like This One

**Code Example 13.16.** The five messages most similar to one complaint.

In [ ]:
# A message Ana flagged while reading
i = msgs.index[msgs["text"].str.contains(
    "NorthPay took my payment twice")][0]
print(msgs.loc[i, "text"][:60])

# Similarity of this message with every other one
sims = pd.Series(cosine_similarity(W[i], W).ravel(),
                 index=msgs.index).drop(i)
for j, s in sims.sort_values(ascending=False).head(5).items():
    print(round(s, 2), "|", msgs.loc[j, "date"].date(), "|",
          msgs.loc[j, "text"][:42])

# How many resemble it, and in which months
close = sims[sims > 0.3].index
print(len(close))
print(msgs.loc[close, "date"].dt.to_period("M").value_counts()
      .sort_index().to_string())

## 13.5 Topics: What Customers Write About

### 13.5.1 Non-negative Matrix Factorization

**Code Example 13.17.** Ten topics from the support messages.

In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Greetings and sign-offs, added to the standard stop words
greet = ["hi", "hello", "hey", "dear", "team", "support", "thanks",
         "thank", "regards", "please", "pls", "help", "advance",
         "appreciate", "morning", "afternoon", "northstar", "let",
         "know", "waiting", "answer", "concern", "advise"]
stop = list(ENGLISH_STOP_WORDS.union(greet))
tfidf = TfidfVectorizer(stop_words=stop, min_df=5)
W = tfidf.fit_transform(msgs["text"])

nmf = NMF(n_components=10, init="nndsvda", max_iter=500,
          random_state=0)
D = nmf.fit_transform(W)          # messages x topics
T = nmf.components_               # topics x words
print(W.shape, D.shape, T.shape)

# The seven heaviest words of each topic
vocab = tfidf.get_feature_names_out()
for k in range(10):
    print(k, " ".join(vocab[T[k].argsort()[::-1][:7]]))

### 13.5.2 Topics Over Time

**Code Example 13.18.** Each message's main topic, and the topic shares by month.

In [ ]:
names = ["setup", "plan_change", "outage", "velofiber", "northpay",
         "slow", "praise", "price", "credit_letter", "moving"]
msgs["topic"] = D.argmax(axis=1)
msgs["topic_name"] = [names[k] for k in msgs["topic"]]

share = pd.crosstab(month, msgs["topic_name"], normalize="index")
cols = ["northpay", "velofiber", "outage", "price"]
print(share[cols].iloc[5:12].round(3).to_string())

**Figure 13.3 — share of each month's messages by main topic.** Not a code example in the printed chapter: this cell draws the figure from the variables above.

In [ ]:
# Percent of each month's messages whose main topic is each topic
pct = pd.crosstab(month, msgs["topic_name"], normalize="index") * 100
x = pct.index.to_timestamp()

# Four topics in color, the other six in light gray
highlight = {"northpay": (BLUE, "NorthPay double charges"),
             "velofiber": ("#E39B2C", "VeloFiber offers"),
             "outage": ("#555555", "outages"),
             "credit_letter": ("#9A9A9A", "credit-request letter")}

fig, ax = plt.subplots(figsize=(10, 4.3))
for t in pct.columns:
    if t not in highlight:
        ax.plot(x, pct[t], color="#D0D0D0", lw=1.2)
for t, (color, label) in highlight.items():
    ax.plot(x, pct[t], color=color, lw=2.4, marker="o", ms=3.5)

# Labels next to the highlighted lines
ax.text(pd.Timestamp("2024-04-20"), 44, "NorthPay double charges", color=BLUE, fontsize=9)
ax.text(pd.Timestamp("2024-12-15"), pct["velofiber"].iloc[-1] + 3, "VeloFiber offers",
        color="#E39B2C", fontsize=9)
ax.annotate("outages", xy=(x[3], pct["outage"].iloc[3]),
            xytext=(pd.Timestamp("2023-11-20"), 38), color="#555555", fontsize=9,
            arrowprops=dict(arrowstyle="-", color="#555555", lw=0.8))
ax.annotate("credit-request letter", xy=(x[3], pct["credit_letter"].iloc[3]),
            xytext=(pd.Timestamp("2023-07-01"), 29), color="#777777", fontsize=9,
            arrowprops=dict(arrowstyle="-", color="#999999", lw=0.8))
ax.plot([], [], color="#D0D0D0", lw=1.2, label="the other six topics")
ax.legend(frameon=False, loc="upper right", fontsize=9)

ticks = pd.date_range("2023-07-01", "2024-12-01", freq="3MS")
ax.set_xticks(ticks)
ax.set_xticklabels([t.strftime("%b %Y") for t in ticks])
ax.set_ylabel("% of the month's messages")
ax.set_ylim(0, 52)
ax.set_xlim(x[0] - pd.Timedelta(days=15), x[-1] + pd.Timedelta(days=110))
ax.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
plt.show()

### 13.5.3 Topics and Churn

**Code Example 13.19.** Churn by topic.

In [ ]:
by_topic = msgs.groupby("topic_name")["churned"].agg(["size", "mean"])
print(by_topic.sort_values("mean").round(3).to_string())

**Figure 13.4 — share of messages in each topic whose author later churned.** Not a code example in the printed chapter: this cell draws the figure from the variables above.

In [ ]:
# Readable names for the topics, in the order of the churn rate
labels = {"setup": "setup and equipment", "plan_change": "plan changes",
          "outage": "outages", "velofiber": "VeloFiber offers",
          "northpay": "NorthPay double charges", "slow": "slow internet",
          "praise": "praise", "price": "prices and fees",
          "credit_letter": "credit-request letter", "moving": "moving"}
rates = by_topic.sort_values("mean")

fig, ax = plt.subplots(figsize=(10, 4.2))
colors = [BLUE if r > 0.5 else "#9A9A9A" for r in rates["mean"]]
ax.barh([labels[t] for t in rates.index], rates["mean"] * 100, color=colors)

# Churn rate and number of messages at the end of each bar
for i, (r, n) in enumerate(zip(rates["mean"], rates["size"])):
    ax.text(r * 100 + 0.8, i, f"{r * 100:.0f}%  ({n:,} messages)",
            va="center", fontsize=8.5, color="#444444")

# Churn rate of all customers, for reference
base = customers["churned"].mean() * 100
ax.axvline(base, color="#333333", ls="--", lw=1)
ax.text(base + 0.4, -0.9, f"all customers: {base:.0f}%", fontsize=8.5, color="#333333")

ax.set_xlabel("% of messages whose author later churned")
ax.set_xlim(0, 80)
ax.set_ylim(-1.2, 9.6)
ax.grid(axis="y", visible=False)
sns.despine()
plt.tight_layout()
plt.show()

### 13.5.4 Checking the Topics Against a Person

**Code Example 13.20.** Topics against the reasons coded by hand.

In [ ]:
tags = pd.read_csv(BASE + "ch13_message_tags.csv")

# Ana's topic names translated into the quality team's reason codes
code = {"setup": "setup", "plan_change": "plan change",
        "outage": "outage", "credit_letter": "outage",
        "velofiber": "competitor offer", "northpay": "billing error",
        "slow": "slow internet", "praise": "praise",
        "price": "price", "moving": "moving"}
v = tags.merge(msgs[["message_id", "topic_name"]], on="message_id")
v["match"] = v["topic_name"].map(code) == v["reason"]
print(len(v), round(v["match"].mean(), 3))
print(v.groupby("reason")["match"].agg(["size", "mean"])
      .round(2).to_string())

### 13.5.5 The Classical Alternative: Latent Dirichlet Allocation

**Code Example 13.21.** Ten LDA topics from the same messages.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

counts = CountVectorizer(stop_words=stop, min_df=5)
C = counts.fit_transform(msgs["text"])
lda = LatentDirichletAllocation(n_components=10, random_state=0,
                                learning_method="batch", max_iter=20)
theta = lda.fit_transform(C)       # topic shares of each message
cw = counts.get_feature_names_out()
for k in range(10):
    print(k, " ".join(cw[lda.components_[k].argsort()[::-1][:7]]))

## 13.6 Use: What the Inbox Adds to the Churn Model

### 13.6.1 From Messages to Customers

**Code Example 13.22.** Text measures for each customer, joined to the churn table.

In [ ]:
# Topic shares and complaint counts, averaged over each customer
topics = pd.DataFrame(D, columns=names)
topics["customer_id"] = msgs["customer_id"].values
text_feats = topics.groupby("customer_id").mean()
text_feats["complaints"] = (msgs.groupby("customer_id")
                            ["complaints"].mean())
text_feats["wrote"] = 1.0

# The variables of the churn table
cust = customers.set_index("customer_id")
X_base = pd.get_dummies(
    cust[["contract_type", "payment_method", "internet_service",
          "paperless_billing"]], drop_first=True).astype(float)
X_base["tenure_months"] = cust["tenure_months"]
X_base["monthly_charges"] = cust["monthly_charges"]
X_base["num_support_calls"] = cust["num_support_calls"].fillna(0)

# The same variables plus the text measures; zeros for non-writers
X_text = X_base.join(text_feats).fillna(0)
y = cust["churned"]
print(X_base.shape, X_text.shape)

### 13.6.2 Comparing the Two Models

**Code Example 13.23.** Cross-validated AUC with and without the messages.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=5000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
p_base = cross_val_predict(model, X_base, y, cv=cv,
                           method="predict_proba")[:, 1]
p_text = cross_val_predict(model, X_text, y, cv=cv,
                           method="predict_proba")[:, 1]
print("AUC, churn table only: ", round(roc_auc_score(y, p_base), 3))
print("AUC, with the messages:", round(roc_auc_score(y, p_text), 3))

wrote = (X_text["wrote"] == 1).to_numpy()
print("customers who wrote:",
      round(roc_auc_score(y[wrote], p_base[wrote]), 3),
      round(roc_auc_score(y[wrote], p_text[wrote]), 3))

**Code Example 13.24.** Churners among the 10 percent of customers ranked riskiest.

In [ ]:
n = int(len(y) * 0.10)                       # 704 calls a month
for name, p in [("churn table only", p_base),
                ("with the messages", p_text)]:
    top = np.argsort(-p)[:n]                 # the n riskiest
    print(name, ":", y.iloc[top].sum(), "churners among", n)

**Code Example 13.25.** Coefficients of the text measures.

In [ ]:
model.fit(X_text, y)
coef = pd.Series(model[-1].coef_[0], index=X_text.columns)
print(coef[names + ["complaints", "wrote"]].sort_values()
      .round(2).to_string())

### 13.6.3 What Ana Tells Helen

## Exercises

Work through each exercise in its own cell below.

**Exercise 1.** *(Counting.)* Repeat Code Example 13.7 for the words *refund* and *slow*. Which of the two moves with the NorthPay episode, and why does the other not?

In [ ]:
# Exercise 1


**Exercise 2.** *(Copies.)* Remove the 134 copies of the form letter and recompute the topic shares of Code Example 13.18. How much does the outage topic change in October and November 2023?

In [ ]:
# Exercise 2


**Exercise 3.** *(Dictionaries.)* Add five terms to the complaint list of Code Example 13.11, chosen by reading messages from the NorthPay and VeloFiber topics. Does the gap between churners and non-churners widen? Check, using `ch13_message_tags.csv`, whether the new terms also appear in messages whose reason is not a complaint.

In [ ]:
# Exercise 3


**Exercise 4.** *(Sentiment.)* Add each customer's average VADER score to the model of Code Example 13.23. Does it improve the AUC once the topics and the complaint count are in the model?

In [ ]:
# Exercise 4


**Exercise 5.** *(Number of topics.)* Fit NMF with 6 and with 15 topics. Which topics of Code Example 13.17 split, which merge, and does the churn rate of the NorthPay topic change?

In [ ]:
# Exercise 5


**Exercise 6.** *(Validation.)* The setup topic matched only half of the hand-coded setup messages. Add *new* to the stop words, refit the model and repeat Code Example 13.20. What changes?

In [ ]:
# Exercise 6


**Exercise 7.** *(LDA.)* Assign each message to its heaviest LDA topic and repeat the check of Code Example 13.20 with your own translation of the LDA topics into reason codes. How does its agreement compare with NMF's 89 percent?

In [ ]:
# Exercise 7


**Exercise 8.** *(A planted event.)* The script `ch13_generate_messages.py` creates the messages from hidden reasons and three planted events. Add a fourth event — for instance, a price increase announced in September 2024 — generate the data again, and check which of the chapter's methods find it and how soon.

In [ ]:
# Exercise 8
